# External Validation: CLAM-MB + UNI2-h PAM50 Pipeline

This report evaluates the **CLAM-MB + UNI2-h** pipeline on three external cohorts, **without retraining or domain adaptation**:

- **CPTAC-BRCA** -- 114 patients / 378 H\&E `.svs` WSIs, **genomic PAM50** labels (Krug et al. 2020, cBioPortal `brca_cptac_2020`). Headline external result.
- **HSI-BC** -- 47 patients, one H\&E `.mrxs` WSI per patient, IHC-derived subtype labels.
- **NOU** -- 28 patients, 203 tissue-block H\&E crops total, IHC-derived subtype labels.

The 10 CLAM-MB fold-checkpoints from TCGA-BRCA are reused as-is. Ensemble predictions average softmax probabilities across all 10 folds.

**PAM50 classes:** LumA (0), LumB (1), Basal (2), Her2 (3).

## Pipeline Overview

The pipeline is identical to the one in `evaluate_pam50.pdf`:

1. **Data preparation** -- subtype labels mapped to the 4 PAM50 classes.
2. **Tissue patching** -- HSI-BC: 256px patches from `.mrxs`; NOU: 512px patches from TIFF crops (both via CLAM `create_patches_fp.py`). CPTAC-BRCA: **not re-patched here** -- the pre-computed UNI2-h features published by MahmoodLab (`MahmoodLab/UNI2-h-features`, `CPTAC/cptac_brca.tar.gz`) are used directly.
3. **Feature extraction** -- UNI2-h (ViT-Giant/14, 1536-dim embeddings).
4. **Ensemble inference** -- 10 TCGA-BRCA CLAM-MB checkpoints, mean-softmax ensemble. No fine-tuning or domain adaptation.

CPTAC-BRCA is the cleanest of the three comparisons: its features come from the **same Trident 20x / 256px / 0px-overlap pipeline and the same UNI2-h weights** that produced the TCGA-BRCA training features, so preprocessing is held constant between train and test. Any drop measured on CPTAC is therefore attributable to cohort and domain shift rather than to a patching or encoder mismatch.

### Label provenance: genomic vs. IHC surrogate

The three cohorts are **not** labelled the same way, and this bounds how far their numbers can be compared.

- **CPTAC-BRCA** carries **genomic PAM50 calls** -- the same class definition the model was trained on (TCGA-BRCA PAM50 from expression data). Label noise is negligible relative to the other two cohorts, so the CPTAC numbers measure *model* transfer.
- **HSI-BC** and **NOU** carry **IHC surrogate** subtypes (ER/PR/HER2/Ki-67 decision rules). Surrogate-to-genomic concordance is roughly 80%, i.e. 10--20% of their "errors" are label-definition disagreements, not model errors -- and the disagreement concentrates exactly where the model is weakest (LumA vs. LumB).

CPTAC is therefore the result the thesis leads with; HSI-BC and NOU are supporting evidence with a noise floor.

## Experimental setup

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc as calc_auc,
)
from sklearn.preprocessing import label_binarize

PALETTE = ["#2563EB", "#DC2626", "#16A34A", "#D97706"]
CLASS_NAMES = ["LumA", "LumB", "Basal", "Her2"]
N_CLASSES = 4
N_FOLDS = 10
CLASS_COLORS = dict(zip(CLASS_NAMES, PALETTE))
PROB_COLS = ["p_LumA", "p_LumB", "p_Basal", "p_Her2"]

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

In [ ]:
DATASETS = {
    "CPTAC-BRCA": {
        "results_dir": "../.scratch/cptac_validation/results/predictions",
        "description": "CPTAC-BRCA (.svs WSIs, genomic PAM50 labels)",
        "labels": "genomic",
    },
    "HSI-BC": {
        "results_dir": "../.scratch/hsi_bc_recurrence/results/predictions",
        "description": "HistologyHSI-BC-Recurrence (47 .mrxs WSIs)",
        "labels": "IHC surrogate",
    },
    "NOU": {
        "results_dir": "../.scratch/nou_clam/results/predictions",
        "description": "NOU tissue crops (203 TIFFs from 28 patients)",
        "labels": "IHC surrogate",
    },
}

datasets = {}
for tag, cfg in DATASETS.items():
    df = pd.read_csv(os.path.join(cfg["results_dir"], "ensemble_predictions.csv"))
    datasets[tag] = df
    print(f"[{tag}] loaded {len(df)} ensemble predictions "
          f"from {df['case_id'].nunique()} patients ({cfg['labels']} labels)")

### Cohort Characteristics

In [ ]:
TCGA_DIST = {"LumA": 508, "LumB": 222, "Basal": 173, "Her2": 83}
TCGA_TOTAL = sum(TCGA_DIST.values())

rows = []
rows.append({"Cohort": "TCGA-BRCA (training, 10-fold CV)",
             "Labels": "genomic",
             "Slides": TCGA_TOTAL,
             "Patients": "~930",
             **{c: f"{TCGA_DIST[c]} ({TCGA_DIST[c]/TCGA_TOTAL:.0%})" for c in CLASS_NAMES}})
for tag, df in datasets.items():
    counts = df["true_name"].value_counts().to_dict()
    n = len(df)
    rows.append({"Cohort": f"{tag} (external)",
                 "Labels": DATASETS[tag]["labels"],
                 "Slides": n,
                 "Patients": df["case_id"].nunique(),
                 **{c: f"{counts.get(c, 0)} ({counts.get(c, 0)/n:.0%})" for c in CLASS_NAMES}})

cohort_df = pd.DataFrame(rows)
print("Table 1: Cohort composition by PAM50 class (slide-level)")
print("=" * 110)
print(cohort_df.to_string(index=False))
print("=" * 110)

# Case-level class distribution, since CPTAC and NOU have several slides per case
rows = []
for tag, df in datasets.items():
    per_case = df.drop_duplicates("case_id")["true_name"].value_counts().to_dict()
    n = df["case_id"].nunique()
    rows.append({"Cohort": tag, "Patients": n,
                 **{c: f"{per_case.get(c, 0)} ({per_case.get(c, 0)/n:.0%})" for c in CLASS_NAMES}})
print()
print("Table 1b: Cohort composition by PAM50 class (case-level)")
print("=" * 110)
print(pd.DataFrame(rows).to_string(index=False))
print("=" * 110)

## Per-Fold Ensemble Member Performance

Each of the 10 TCGA-trained checkpoints is applied independently to every external slide (no fold-wise splitting on the external set).

In [ ]:
def per_fold_metrics(results_dir):
    rows = []
    for fold in range(N_FOLDS):
        path = os.path.join(results_dir, f"fold_{fold}_predictions.csv")
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        y = df["true_label"].values
        yp = df["pred_label"].values
        P = df[PROB_COLS].values
        yb = label_binarize(y, classes=list(range(N_CLASSES)))
        try:
            auc = roc_auc_score(yb, P, multi_class="ovr", average="macro")
        except ValueError:
            auc = float("nan")
        rows.append({"Fold": fold,
                     "AUC": auc,
                     "Acc": (y == yp).mean(),
                     "BAcc": balanced_accuracy_score(y, yp)})
    return pd.DataFrame(rows).set_index("Fold")

fold_results = {tag: per_fold_metrics(cfg["results_dir"]) for tag, cfg in DATASETS.items()}

print("Table 2: Per-fold ensemble member metrics on each external cohort")
print("=" * 70)
for tag, df in fold_results.items():
    print(f"--- {tag} ---")
    print(df.to_string(float_format=lambda x: f"{x:.4f}"))
    print(f"  Mean +/- Std: AUC={df['AUC'].mean():.4f} +/- {df['AUC'].std():.4f}  "
          f"Acc={df['Acc'].mean():.4f} +/- {df['Acc'].std():.4f}  "
          f"BAcc={df['BAcc'].mean():.4f} +/- {df['BAcc'].std():.4f}")
    print()
print("=" * 70)

In [ ]:
fig, axes = plt.subplots(1, len(fold_results), figsize=(5.5 * len(fold_results), 3.8),
                         sharey=True, squeeze=False)
for ax, (tag, df) in zip(axes[0], fold_results.items()):
    folds = df.index.values
    width = 0.35
    ax.bar(folds - width/2, df["AUC"].values, width,
           label="Macro AUC", color="#2563EB", edgecolor="white")
    ax.bar(folds + width/2, df["BAcc"].values, width,
           label="Balanced Acc.", color="#16A34A", edgecolor="white")
    mean_auc = df["AUC"].mean()
    ax.axhline(mean_auc, color="#DC2626", ls="--", lw=1.0,
               label=f"mean AUC = {mean_auc:.3f}")
    ax.axhline(0.5, color="gray", ls=":", lw=0.8, alpha=0.7)
    ax.set_title(f"{tag}")
    ax.set_xlabel("Fold")
    ax.set_xticks(folds)
    ax.set_ylim([0.0, 1.0])
    ax.legend(fontsize=8, loc="upper right")
axes[0, 0].set_ylabel("Score")
fig.suptitle("Figure 1: Per-fold ensemble member performance on external cohorts",
             fontweight="bold", y=1.04)
fig.tight_layout()
plt.show()

## Aggregated Ensemble Predictions (Slide-Level)

Mean-softmax ensemble of 10 fold-checkpoints. For HSI-BC, slide-level = patient-level (one WSI per patient). CPTAC-BRCA and NOU have several slides per case, so their slide-level numbers are **not** patient-weighted -- cases contributing more slides count proportionally more. Case-level aggregation follows below and is the number to quote for CPTAC.

In [ ]:
def compute_metrics(df, prob_cols=PROB_COLS):
    y = df["true_label"].values.astype(int)
    yp = df["pred_label"].values.astype(int)
    P = df[prob_cols].values.astype(float)
    yb = label_binarize(y, classes=list(range(N_CLASSES)))
    try:
        macro_auc = roc_auc_score(yb, P, multi_class="ovr", average="macro")
    except ValueError:
        macro_auc = float("nan")
    class_aucs = []
    for i in range(N_CLASSES):
        if yb[:, i].sum() > 0 and yb[:, i].sum() < len(y):
            class_aucs.append(roc_auc_score(yb[:, i], P[:, i]))
        else:
            class_aucs.append(float("nan"))
    report = classification_report(y, yp, target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    return {
        "y": y, "yp": yp, "P": P, "yb": yb,
        "macro_auc": macro_auc,
        "class_aucs": class_aucs,
        "acc": (y == yp).mean(),
        "bacc": balanced_accuracy_score(y, yp),
        "wf1": report["weighted avg"]["f1-score"],
        "mf1": report["macro avg"]["f1-score"],
        "report": report,
    }

image_metrics = {tag: compute_metrics(df) for tag, df in datasets.items()}

print("Table 3: Aggregated ensemble metrics (slide-level)")
print("=" * 78)
header = f"  {'Cohort':<10s}  {'Macro AUC':>10s}  {'Acc':>8s}  {'BAcc':>8s}  {'Macro F1':>10s}  {'Wt. F1':>8s}"
print(header)
print("  " + "-" * 64)
for tag, m in image_metrics.items():
    print(f"  {tag:<10s}  {m['macro_auc']:>10.4f}  {m['acc']:>8.4f}  "
          f"{m['bacc']:>8.4f}  {m['mf1']:>10.4f}  {m['wf1']:>8.4f}")
print("=" * 78)
print()
print("Note: BAcc (balanced accuracy) is the macro-averaged recall and equals")
print("random-chance level (1/4 = 0.25) when the model collapses to a single class.")

In [ ]:
for tag, m in image_metrics.items():
    print(f"Classification report -- {tag} (slide-level)")
    print("=" * 60)
    print(classification_report(m["y"], m["yp"],
                                target_names=CLASS_NAMES, zero_division=0))
    print()

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(len(image_metrics), 2, figsize=(11, 4.5 * len(image_metrics)),
                         squeeze=False)
for row, (tag, m) in enumerate(image_metrics.items()):
    cm_raw = confusion_matrix(m["y"], m["yp"], labels=list(range(N_CLASSES)))
    row_sums = cm_raw.sum(axis=1, keepdims=True).astype(float)
    row_sums[row_sums == 0] = 1
    cm_norm = cm_raw / row_sums

    ConfusionMatrixDisplay(cm_raw, display_labels=CLASS_NAMES).plot(
        ax=axes[row, 0], cmap="Blues", colorbar=False)
    axes[row, 0].set_title(f"{tag} -- counts")
    ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
        ax=axes[row, 1], cmap="Blues", colorbar=False, values_format=".2f")
    axes[row, 1].set_title(f"{tag} -- row-normalised (recall)")
    for a in axes[row]:
        a.set_xlabel("Predicted")
        a.set_ylabel("True")

fig.suptitle("Figure 2: Slide-level confusion matrices on external cohorts",
             fontweight="bold", y=1.00)
fig.tight_layout()
plt.show()

### ROC Curves (One-vs-Rest)

In [ ]:
fig, axes = plt.subplots(1, len(image_metrics), figsize=(5.5 * len(image_metrics), 5),
                         squeeze=False)
for ax, (tag, m) in zip(axes[0], image_metrics.items()):
    for i, name in enumerate(CLASS_NAMES):
        if m["yb"][:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(m["yb"][:, i], m["P"][:, i])
        auc_val = calc_auc(fpr, tpr)
        ax.plot(fpr, tpr, color=CLASS_COLORS[name], lw=2,
                label=f"{name} (AUC = {auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.4)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{tag}")
    ax.legend(fontsize=9, loc="lower right")
    ax.set_aspect("equal")
fig.suptitle("Figure 3: One-vs-rest ROC curves on external cohorts",
             fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

### Per-class breakdown

In [ ]:
rows = []
for tag, m in image_metrics.items():
    for i, name in enumerate(CLASS_NAMES):
        r = m["report"][name]
        rows.append({
            "Cohort": tag,
            "Class": name,
            "Support": int(r["support"]),
            "AUC": m["class_aucs"][i],
            "Precision": r["precision"],
            "Recall": r["recall"],
            "F1": r["f1-score"],
        })
perclass_df = pd.DataFrame(rows)
print("Table 4: Per-class metrics on external cohorts (slide-level)")
print("=" * 78)
print(perclass_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("=" * 78)

## NOU Patient-Level Evaluation

We aggregate the ensemble softmax probabilities across all tissue blocks of a patient by averaging, then argmax for the patient-level prediction.

In [ ]:
nou = datasets["NOU"]
patient_df = (nou.groupby("case_id")
                 .agg(true_label=("true_label", "first"),
                      true_name=("true_name", "first"),
                      p_LumA=("p_LumA", "mean"),
                      p_LumB=("p_LumB", "mean"),
                      p_Basal=("p_Basal", "mean"),
                      p_Her2=("p_Her2", "mean"),
                      n_blocks=("slide_id", "count"))
                 .reset_index())
patient_df["pred_label"] = patient_df[PROB_COLS].values.argmax(axis=1)
patient_df["pred_name"] = patient_df["pred_label"].map(dict(enumerate(CLASS_NAMES)))

patient_metrics_nou = compute_metrics(patient_df)

print(f"NOU patient-level evaluation: n = {len(patient_df)} patients, "
      f"median {patient_df['n_blocks'].median():.0f} blocks/patient")
print("=" * 60)
print(f"  Macro AUC:           {patient_metrics_nou['macro_auc']:.4f}")
print(f"  Accuracy:            {patient_metrics_nou['acc']:.4f}")
print(f"  Balanced accuracy:   {patient_metrics_nou['bacc']:.4f}")
print(f"  Macro F1:            {patient_metrics_nou['mf1']:.4f}")
print(f"  Weighted F1:         {patient_metrics_nou['wf1']:.4f}")
print()
print(classification_report(patient_metrics_nou["y"], patient_metrics_nou["yp"],
                            target_names=CLASS_NAMES, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
cm_raw = confusion_matrix(patient_metrics_nou["y"], patient_metrics_nou["yp"],
                          labels=list(range(N_CLASSES)))
row_sums = cm_raw.sum(axis=1, keepdims=True).astype(float)
row_sums[row_sums == 0] = 1
cm_norm = cm_raw / row_sums

ConfusionMatrixDisplay(cm_raw, display_labels=CLASS_NAMES).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Counts")
ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Row-normalised (recall)")
for a in axes:
    a.set_xlabel("Predicted")
    a.set_ylabel("True")
fig.suptitle("Figure 4: NOU patient-level confusion matrix (n=28)",
             fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## CPTAC-BRCA Case-Level Evaluation

CPTAC-BRCA contributes up to 7 slides per case (median 3), so the slide-level table above over-weights heavily-sampled cases. We aggregate to the patient by **averaging the ensemble softmax across all slides of a case**, then taking the argmax -- the same rule used for NOU above, and the natural one here because every slide of a case shares one genomic PAM50 call and no slide is privileged a priori (CPTAC ships multiple tissue blocks per patient, not one designated diagnostic slide as TCGA mostly does).

The case-level number is the one to quote against TCGA's patient-level CV.

In [ ]:
def aggregate_cases(df, count_col="n_slides"):
    """Mean-softmax across all slides of a case, then argmax."""
    agg = (df.groupby("case_id")
             .agg(true_label=("true_label", "first"),
                  true_name=("true_name", "first"),
                  p_LumA=("p_LumA", "mean"),
                  p_LumB=("p_LumB", "mean"),
                  p_Basal=("p_Basal", "mean"),
                  p_Her2=("p_Her2", "mean"),
                  **{count_col: ("slide_id", "count")})
             .reset_index())
    agg["pred_label"] = agg[PROB_COLS].values.argmax(axis=1)
    agg["pred_name"] = agg["pred_label"].map(dict(enumerate(CLASS_NAMES)))
    return agg

cptac_case_df = aggregate_cases(datasets["CPTAC-BRCA"])
cptac_case_metrics = compute_metrics(cptac_case_df)

print(f"CPTAC-BRCA case-level evaluation: n = {len(cptac_case_df)} patients, "
      f"{len(datasets['CPTAC-BRCA'])} slides "
      f"(median {cptac_case_df['n_slides'].median():.0f}, "
      f"max {cptac_case_df['n_slides'].max():.0f} slides/patient)")
print("=" * 60)
print(f"  Macro AUC:           {cptac_case_metrics['macro_auc']:.4f}")
print(f"  Accuracy:            {cptac_case_metrics['acc']:.4f}")
print(f"  Balanced accuracy:   {cptac_case_metrics['bacc']:.4f}")
print(f"  Macro F1:            {cptac_case_metrics['mf1']:.4f}")
print(f"  Weighted F1:         {cptac_case_metrics['wf1']:.4f}")
print()
print("Per-class one-vs-rest AUC:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<6s} {cptac_case_metrics['class_aucs'][i]:.4f}")
print()
print(classification_report(cptac_case_metrics["y"], cptac_case_metrics["yp"],
                            target_names=CLASS_NAMES, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
cm_raw = confusion_matrix(cptac_case_metrics["y"], cptac_case_metrics["yp"],
                          labels=list(range(N_CLASSES)))
row_sums = cm_raw.sum(axis=1, keepdims=True).astype(float)
row_sums[row_sums == 0] = 1
cm_norm = cm_raw / row_sums

ConfusionMatrixDisplay(cm_raw, display_labels=CLASS_NAMES).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Counts")
ConfusionMatrixDisplay(cm_norm, display_labels=CLASS_NAMES).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Row-normalised (recall)")
for a in axes:
    a.set_xlabel("Predicted")
    a.set_ylabel("True")
fig.suptitle(f"Figure 5: CPTAC-BRCA case-level confusion matrix (n={len(cptac_case_df)})",
             fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## CPTAC-BRCA: Multimodal External Validation (WSI + RNA)

The same 114 CPTAC cases carry RNA-seq, so the gated-fusion head can be validated on exactly the cohort used above -- no subset shrinkage, and the WSI-only row is a like-for-like baseline.

Fusion needs the two cohorts' expression on one scale, and there are two ways to get there. They are reported side by side because the choice is itself a methodological result.

**Tier 1 -- one quantification pipeline (primary).** TCGA-BRCA RNA is re-derived from GDC `STAR - Counts` (`tools/rna/download_gdc_rna.py`), the same workflow, genome build and GENCODE v36 annotation that produced the CPTAC files, then both are collapsed to log2(TPM+1) on a shared Ensembl gene axis (`tools/rna/build_gdc_expression.py`). **No cross-cohort normalisation is applied at all.** The RNA branch of the fusion head is retrained on this table; the WSI branch stays frozen at the same `pam50_final_s1` checkpoints used for the WSI-only result above.

**Tier 2 -- FSQN onto the original Xena scale (sensitivity).** The fusion head exactly as published in the thesis, trained on Xena HiSeqV2 (log2 RSEM, HUGO symbols). CPTAC is mapped onto that scale with Feature Specific Quantile Normalization (Franks, Cai & Whitfield, *Bioinformatics* 2018, `10.1093/bioinformatics/bty026`; re-validated for breast PAM50 by Skubleny et al. 2024, `10.1186/s12859-024-05759-w`), after resolving deprecated gene symbols through the HGNC previous-symbol table. FSQN uses the *target* cohort's per-gene distribution, so unlike Tier 1 it is unsupervised domain adaptation and is a weaker claim.

**RNA-only** is the attribution control: it says how much of the fusion result the RNA branch carries alone. Because PAM50 is defined from expression, it is also the harmonisation check -- a collapse there would mean the scales are wrong, not that the model is.

In [ ]:
MODALITIES = {
    "WSI only": {
        "dir": "../.scratch/cptac_validation/results/predictions",
        "rna": "--",
        "note": "CLAM-MB, UNI2-h features, frozen",
    },
    "RNA only": {
        "dir": "../.scratch/cptac_validation/results/predictions_rna",
        "rna": "Tier 1 (GDC STAR, no normalisation)",
        "note": "RNA MLP, frozen",
    },
    "Fusion (Tier 1)": {
        "dir": "../.scratch/cptac_validation/results/predictions_fusion",
        "rna": "Tier 1 (GDC STAR, no normalisation)",
        "note": "gated fusion, WSI branch frozen",
    },
    "Fusion (Tier 2, FSQN)": {
        "dir": "../.scratch/cptac_validation/results/predictions_fusion_fsqn",
        "rna": "Tier 2 (FSQN onto Xena)",
        "note": "gated fusion as published, frozen",
    },
    "Fusion (Tier 1, RNA ablated)": {
        "dir": "../.scratch/cptac_validation/results/predictions_fusion_ablate",
        "rna": "RNA held at training mean (z=0)",
        "note": "control: fusion head with no RNA signal",
    },
}

def case_level(df):
    """Mean ensemble softmax within a case, then argmax."""
    if df["case_id"].nunique() == len(df):
        return df.copy()
    agg = (df.groupby("case_id")
             .agg(true_label=("true_label", "first"),
                  **{c: (c, "mean") for c in PROB_COLS})
             .reset_index())
    agg["pred_label"] = agg[PROB_COLS].to_numpy().argmax(axis=1)
    return agg

mm = {}
for tag, cfg in MODALITIES.items():
    path = os.path.join(cfg["dir"], "ensemble_predictions.csv")
    if not os.path.exists(path):
        print(f"[{tag}] not found at {path} -- skipping")
        continue
    df = pd.read_csv(path)
    mm[tag] = {"slide": df, "case": case_level(df), **cfg}
    print(f"[{tag}] {len(df)} rows, {df['case_id'].nunique()} cases")

In [ ]:
# TCGA internal 10-fold CV, pooled across folds. WSI-only and both fusion rows come
# from split_*_results.pkl; RNA-only from fold_*_test_predictions.csv. All recomputed
# in this session rather than quoted, so the pooling is identical across rows.
TCGA_INTERNAL = {
    "WSI only":              {"Macro AUROC": 0.8879, "Balanced Acc.": 0.6552},
    "RNA only":              {"Macro AUROC": 0.9766, "Balanced Acc.": 0.8678},
    "Fusion (Tier 1)":       {"Macro AUROC": 0.9721, "Balanced Acc.": 0.8468},
    "Fusion (Tier 2, FSQN)": {"Macro AUROC": 0.9744, "Balanced Acc.": 0.8724},
}

rows = []
for tag, entry in mm.items():
    m = compute_metrics(entry["case"])
    internal = TCGA_INTERNAL.get(tag, {})
    rows.append({
        "Model": tag,
        "RNA harmonisation": entry["rna"],
        "Macro AUROC": m["macro_auc"],
        "Balanced Acc.": m["bacc"],
        "Accuracy": m["acc"],
        "LumA": m["class_aucs"][0],
        "LumB": m["class_aucs"][1],
        "Basal": m["class_aucs"][2],
        "Her2": m["class_aucs"][3],
        "Her2 recall": m["report"]["Her2"]["recall"],
        "TCGA int. AUROC": internal.get("Macro AUROC", float("nan")),
    })

mm_df = pd.DataFrame(rows)
print("Table 6: CPTAC-BRCA case-level external validation by modality (n = 114 patients)")
print("=" * 150)
print(mm_df.to_string(index=False, float_format=lambda x: "  -- " if pd.isna(x) else f"{x:.4f}"))
print("=" * 150)
print()
print("Per-class columns are one-vs-rest AUROC. Case level = mean ensemble softmax within a case.")
print("TCGA int. AUROC is the internal 10-fold cross-validation figure for the same model.")

In [ ]:
n = len(mm)
fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.6), squeeze=False)
for ax, (tag, entry) in zip(axes[0], mm.items()):
    m = compute_metrics(entry["case"])
    cm_raw = confusion_matrix(m["y"], m["yp"], labels=list(range(N_CLASSES)))
    row_sums = cm_raw.sum(axis=1, keepdims=True).astype(float)
    row_sums[row_sums == 0] = 1
    ConfusionMatrixDisplay(cm_raw / row_sums, display_labels=CLASS_NAMES).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(tag, fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
fig.suptitle("Figure 6: CPTAC-BRCA case-level confusion matrices by modality (row-normalised)",
             fontweight="bold", y=1.05)
fig.tight_layout()
plt.show()

In [ ]:
axis = pd.read_csv("../.scratch/rna-gdc/gene_axis.csv")
print("Harmonisation evidence -- Tier 1 (one GDC pipeline, no normalisation applied)")
print("=" * 78)
print(f"  shared protein-coding genes        {len(axis)}")
print(f"  correlation of per-gene means      {np.corrcoef(axis['tcga_mean'], axis['cptac_mean'])[0, 1]:.4f}")
print(f"  median per-gene offset (CPTAC-TCGA) {(axis['cptac_mean'] - axis['tcga_mean']).median():+.4f} log2 units")
print(f"  genes the model needs but CPTAC lacks   0")
print()
print("For contrast, Tier 2 maps across quantification pipelines: of the 10,000 genes each")
print("fold selected from Xena, ~890 (8.9%) have no CPTAC counterpart even after HGNC")
print("previous-symbol resolution, and are imputed at the training mean.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(axis["tcga_mean"], axis["cptac_mean"], s=1, alpha=0.15, color="#2563EB")
lim = [0, max(axis["tcga_mean"].max(), axis["cptac_mean"].max())]
axes[0].plot(lim, lim, "k--", lw=0.8)
axes[0].set_xlabel("TCGA mean log2(TPM+1)")
axes[0].set_ylabel("CPTAC mean log2(TPM+1)")
axes[0].set_title("Per-gene means, Tier 1")
axes[0].set_aspect("equal")

axes[1].hist(axis["cptac_mean"] - axis["tcga_mean"], bins=120, color="#16A34A")
axes[1].axvline(0, color="k", ls="--", lw=0.8)
axes[1].set_xlabel("CPTAC - TCGA mean log2(TPM+1)")
axes[1].set_ylabel("genes")
axes[1].set_title("Per-gene offset, Tier 1")

fig.suptitle("Figure 7: cross-cohort expression agreement under one quantification pipeline",
             fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## Comparison: Internal vs. External

In [ ]:
tcga_summary = {
    "Macro AUC": 0.8879,
    "Accuracy": 0.7181,
    "Balanced Acc.": 0.6552,   # recomputed from .scratch/results/pam50_final_s1/split_*_results.pkl
    "Weighted F1": 0.7179,
    "LumA AUC": 0.8967,
    "LumB AUC": 0.8174,
    "Basal AUC": 0.9529,
    "Her2 AUC": 0.8846,
}

def to_row(m):
    return {
        "Macro AUC": m["macro_auc"],
        "Accuracy": m["acc"],
        "Balanced Acc.": m["bacc"],
        "Weighted F1": m["wf1"],
        "LumA AUC": m["class_aucs"][0],
        "LumB AUC": m["class_aucs"][1],
        "Basal AUC": m["class_aucs"][2],
        "Her2 AUC": m["class_aucs"][3],
    }

comp = pd.DataFrame({
    "TCGA-BRCA (10-fold CV)": tcga_summary,
    "CPTAC (slide-level)": to_row(image_metrics["CPTAC-BRCA"]),
    "CPTAC (case-level)": to_row(cptac_case_metrics),
    "HSI-BC (slide=patient)": to_row(image_metrics["HSI-BC"]),
    "NOU (slide-level)": to_row(image_metrics["NOU"]),
    "NOU (patient-level)": to_row(patient_metrics_nou),
})

print("Table 5: TCGA-BRCA internal vs. external cohorts")
print("=" * 120)
print(comp.to_string(float_format=lambda x: "  -- " if pd.isna(x) else f"{x:.4f}"))
print("=" * 120)
print()
print("TCGA values are reproduced from evaluate_pam50.pdf (10-fold CV, 986 slides pooled across folds);")
print("balanced accuracy was recomputed from the same split_*_results.pkl -- its macro/per-class AUCs")
print("reproduce the published values exactly, confirming the pooled set is identical.")
print("CPTAC uses genomic PAM50 labels; HSI-BC and NOU use IHC surrogates (~80% concordance).")

## Findings

### WSI track

1. **CPTAC-BRCA: the H\&E model transfers.** Case-level macro AUROC **0.847** vs. **0.888** on TCGA internal CV -- a drop of 4.1 pp on 114 unseen patients, frozen weights, no domain adaptation. Balanced accuracy falls further, 0.655 -> **0.513**: the ranking survives transfer better than the decision rule. CPTAC features come from the same UNI2-h encoder at the same 256px / 20x / 0-overlap geometry as the training features (`tools/cptac/audit_feature_provenance.py`), so this drop is cohort shift, not preprocessing.

2. **Her2 is never predicted from H\&E on CPTAC, yet stays rankable.** Her2 recall is **0.00** at slide and case level -- across all 14 Her2 cases `p_Her2` is the *lowest* of the four probabilities -- while Her2 one-vs-rest AUROC is **0.860**. The failure is in the argmax decision rule, not the representation. On TCGA the same model reaches 0.47 Her2 recall, so this is a calibration shift, and the single largest contributor to the balanced-accuracy gap.

3. **Basal transfers best, LumB worst -- the same ordering as internally.** Case-level per-class AUROC: Basal 0.972, LumA 0.861, Her2 0.860, LumB 0.693, against TCGA's 0.953 / 0.897 / 0.885 / 0.817. Basal *improves*; LumB loses 12 pp.

4. **Case-level beats slide-level.** Macro AUROC 0.816 -> 0.847, balanced accuracy 0.437 -> 0.513 when averaging softmax within a case. CPTAC ships up to 10 slides per case (median 3) and the top 12 cases carry 21% of all slides, so slide-level numbers are not patient-weighted. Case level is the number to quote.

5. **HSI-BC and NOU are bounded by label noise, CPTAC is not.** HSI-BC reaches 0.804 macro AUROC; NOU collapses to chance. Both use IHC surrogates (~80% concordance), so part of their error is label disagreement. NOU's collapse is still best explained by bag size (~210 patches/slide); for reference the 6 CPTAC slides with under 200 patches score 0.50 accuracy against 0.64 for the cohort, but they cover only 5 cases and do not move the headline.

### Multimodal track

6. **Under one quantification pipeline, RNA transfers essentially perfectly.** RNA-only reaches macro AUROC **0.988** and balanced accuracy **0.909** on CPTAC, *above* its own TCGA internal CV (0.977 / 0.868). Zero genes are missing, per-gene means correlate at 0.990 with a median offset of -0.055 log2 units. Cross-cohort expression harmonisation is therefore not a residual confound in anything below -- which is exactly what re-deriving TCGA from GDC bought.

7. **Fusion does not beat RNA alone on CPTAC.** Gated fusion: macro AUROC **0.981**, balanced accuracy **0.888** -- slightly *below* RNA-only (0.988 / 0.909). Adding the H\&E branch to a working RNA branch costs a little rather than adding anything. Fusion does clear its own TCGA internal CV (0.972 / 0.847), so the fusion head itself transfers; it just has nothing to gain from.

8. **The ablation shows the decision is RNA-determined.** Holding RNA at the training mean (z=0) makes the fusion model predict **LumA for all 114 cases** -- balanced accuracy 0.250, exactly chance. The learned gate is nevertheless close to even on CPTAC (WSI 0.462 / RNA 0.538), and the ablated model still ranks at macro AUROC 0.785, so the WSI branch does carry information into the fused logits. It simply never moves the argmax. **A near-balanced gate weight is not evidence of a balanced decision**, and any claim that this architecture "combines" H\&E and RNA has to reckon with that.

9. **The harmonisation route is worth measuring, not assuming.** Tier 1 (one GDC pipeline, no normalisation) gives 0.981 / 0.888; Tier 2 (FSQN onto the Xena scale) gives 0.976 / 0.846, and still leaves ~890 of 10,000 genes per fold (8.9%) imputed at the training mean even after HGNC symbol resolution. FSQN recovers most of the gap a naive scale match would leave, but the pipeline-level fix is both cleaner and measurably better.

10. **Retraining on GDC costs nothing internally.** The GDC-trained fusion scores 0.972 macro AUROC on TCGA internal CV against 0.974 for the Xena-trained original, so restating the thesis's internal fusion numbers in GDC space is a free move.

11. **Read the multimodal numbers with the circularity in mind.** PAM50 is *defined* from expression, so an RNA branch predicting PAM50 is close to tautological -- 0.99 external AUROC is a measure of how well the harmonised expression reproduces the subtype call, not of clinical discovery. The defensible claims here are "the fusion model survives cohort transfer" and "H\&E adds nothing once RNA is present", not "multimodal fusion improves subtyping".

## Next Steps

1. **Her2 decision rule on the WSI branch.** Per-class thresholds or a prior correction fitted on TCGA held-out folds -- never on CPTAC labels -- to test whether Her2's 0.86 H\&E AUROC can be turned into non-zero recall.
2. **Bootstrap confidence intervals** (stratified, 1000 resamples) for every headline metric; 14 Her2 and 17 LumB cases leave the per-class numbers wide, and the fusion-vs-RNA-only gap (0.981 vs 0.988) is well inside what 114 cases can resolve.
3. **A fusion objective that cannot ignore a modality** -- modality dropout during training, or an auxiliary per-branch loss -- since finding 8 shows the current gate lets the RNA branch decide alone. The FiLM and co-attention modes already in `models/model_multimodal.py` are the obvious things to test.
4. **Trident-vs-CLAM tiling ablation** on the WSI side. CPTAC features match the training geometry exactly but were tiled with CLAM `create_patches_fp.py` rather than Trident, so tissue segmentation differs; re-extracting a CPTAC subset with Trident would bound how much of the 4.1 pp WSI drop is tiling rather than cohort.
5. **A task where RNA is not the label's own definition** -- recurrence, survival, or treatment response -- if the thesis wants to argue that multimodal fusion adds clinical value rather than reproducing a transcriptomic call.

## Summary

In [ ]:
print("=" * 78)
print("  CLAM-MB + UNI2-h | PAM50 Subtyping | External Validation")
print("  Model: 10x checkpoints from TCGA-BRCA (no retraining, no adaptation)")
print("=" * 78)
print()
print("--- TCGA-BRCA internal 10-fold CV ---")
for k, v in tcga_summary.items():
    if pd.isna(v):
        continue
    print(f"  {k:<14s}  {v:.4f}")
print()
for tag, m in image_metrics.items():
    print(f"--- {tag} (slide-level, n = {len(datasets[tag])}) ---")
    print(f"  Macro AUROC    {m['macro_auc']:.4f}")
    print(f"  Balanced Acc.  {m['bacc']:.4f}")
    print()
print(f"--- NOU (patient-level, n = {len(patient_df)}) ---")
print(f"  Macro AUROC    {patient_metrics_nou['macro_auc']:.4f}")
print(f"  Balanced Acc.  {patient_metrics_nou['bacc']:.4f}")
print()
print("=" * 78)
print(f"  CPTAC-BRCA case-level, n = {len(cptac_case_df)} patients (genomic PAM50)")
print("=" * 78)
for tag, entry in mm.items():
    m = compute_metrics(entry["case"])
    print(f"  {tag:<30s} AUROC {m['macro_auc']:.4f}   BAcc {m['bacc']:.4f}   "
          f"Her2 recall {m['report']['Her2']['recall']:.2f}")
print("=" * 78)